In [30]:
import warnings
warnings.filterwarnings('ignore')

In [31]:
import torch
import torch.nn as nn

In [44]:
data = [
    ("I love machine", "learning"),
    ("I love deep", "learning"),
    ("I like machine", "learning"),
    ("I like deep", "learning"),
    ("machine learning is", "powerful"),
    ("deep learning is", "powerful"),
    ("Python is very", "useful"),
    ("PyTorch is very", "useful"),
    ("AI is curcial for out", "task"),
    ("GRU is very", "powerful"),
]

In [45]:
def build_vocab(sentences):
    vocab = {
        "<pad>": 0,
        "<unk>": 1
    }

    for sentence in sentences:

        for word in sentence.split():
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

input_vocab = build_vocab(x[0] for x in data)
target_vocab = build_vocab(x[1] for x in data)

In [46]:
print(input_vocab)
print(target_vocab)

{'<pad>': 0, '<unk>': 1, 'I': 2, 'love': 3, 'machine': 4, 'deep': 5, 'like': 6, 'learning': 7, 'is': 8, 'Python': 9, 'very': 10, 'PyTorch': 11, 'AI': 12, 'curcial': 13, 'for': 14, 'out': 15, 'GRU': 16}
{'<pad>': 0, '<unk>': 1, 'learning': 2, 'powerful': 3, 'useful': 4, 'task': 5}


In [47]:
def text_to_number(sentence, vocab):
    return torch.tensor(
        [vocab.get(word, vocab['<unk>']) for word in sentence.split()],
        dtype=torch.long
    )

input = [text_to_number(sentence, input_vocab) for sentence, _ in data]
output = [text_to_number(sentence, target_vocab) for _, sentence in data]

In [48]:
print(input), print(output)

[tensor([2, 3, 4]), tensor([2, 3, 5]), tensor([2, 6, 4]), tensor([2, 6, 5]), tensor([4, 7, 8]), tensor([5, 7, 8]), tensor([ 9,  8, 10]), tensor([11,  8, 10]), tensor([12,  8, 13, 14, 15]), tensor([16,  8, 10])]
[tensor([2]), tensor([2]), tensor([2]), tensor([2]), tensor([3]), tensor([3]), tensor([4]), tensor([4]), tensor([5]), tensor([3])]


(None, None)

In [49]:
from torch.nn.utils.rnn import pad_sequence

input_pad = pad_sequence(
    input,
    batch_first=True,
    padding_value=input_vocab["<pad>"]
)

print(input_pad)

tensor([[ 2,  3,  4,  0,  0],
        [ 2,  3,  5,  0,  0],
        [ 2,  6,  4,  0,  0],
        [ 2,  6,  5,  0,  0],
        [ 4,  7,  8,  0,  0],
        [ 5,  7,  8,  0,  0],
        [ 9,  8, 10,  0,  0],
        [11,  8, 10,  0,  0],
        [12,  8, 13, 14, 15],
        [16,  8, 10,  0,  0]])


In [53]:
vocab_size = len(input_vocab)
embedding_dim = 10

embedding = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=embedding_dim
)

In [54]:
x = torch.tensor([2, 3, 4])
embedded = embedding(x)
print(embedded)

tensor([[ 0.1433,  0.4621, -1.5184,  0.6614,  0.9563, -0.7032,  0.5145,  0.1287,
         -0.1454, -1.9295],
        [ 0.0325, -1.2498,  0.2519, -1.6553, -0.4488,  0.2549,  1.2215,  0.9655,
          0.0933, -1.2068],
        [-1.1606, -0.3003, -0.4396,  0.5604, -0.5138,  0.3047,  1.0688, -1.7896,
          0.3669,  0.9408]], grad_fn=<EmbeddingBackward0>)


In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
        )

        self.gru = nn.GRU(
            input_size=embedding_dim,
            hidden_size=hidden_dim
        )

    def forward(self, sentence):
        embedded = self.embedding(sentence)
        return self.gru(embedded)

In [71]:
model = GRUClassifier(vocab_size, embedding_dim, 32)

In [72]:
x = model(input_pad)

In [73]:
print(len(x))
print(type(x))

2
<class 'tuple'>


In [74]:
output, hidden = x
print(output.shape)

torch.Size([10, 5, 32])


In [75]:
print(hidden)

(tensor([[[ 0.2400, -0.0706, -0.0608, -0.1394,  0.1168,  0.0725,  0.1570,
           0.1012,  0.0105,  0.1320, -0.0081,  0.2639, -0.0630,  0.1017,
           0.4388,  0.1599, -0.0302,  0.0813, -0.1750,  0.1655,  0.1483,
           0.1390,  0.2851,  0.1088,  0.0214,  0.0184, -0.0172,  0.1196,
          -0.0284,  0.0477, -0.0107,  0.0206],
         [-0.0555,  0.0583,  0.0163,  0.1671, -0.0300,  0.1170,  0.2861,
          -0.1859,  0.0674, -0.1507,  0.5936,  0.0290, -0.2466,  0.1773,
          -0.1228,  0.2102,  0.1429, -0.2650,  0.1591, -0.1854,  0.0617,
          -0.0065,  0.1747,  0.1801,  0.2639,  0.1960,  0.1480, -0.2612,
          -0.1726, -0.2545,  0.1234,  0.0098],
         [ 0.1541,  0.0202, -0.0907,  0.1271,  0.0298,  0.0961,  0.2677,
          -0.0707,  0.0338, -0.1090,  0.1924,  0.2479, -0.0681,  0.1265,
           0.1860,  0.1630,  0.1402, -0.0137,  0.0475,  0.0399,  0.1073,
           0.0116,  0.1557,  0.2080, -0.0311,  0.0392,  0.0514, -0.1331,
          -0.1264, -0.0965,  